# BERT Fine-tuning for Financial News Sentiment Analysis

## Project Overview
This notebook demonstrates fine-tuning a pre-trained BERT (Bidirectional Encoder Representations from Transformers) model for sentiment classification on financial news data. BERT has revolutionized NLP by introducing bidirectional context understanding through masked language modeling and is ideal for text classification tasks.

## Key Concepts
- **BERT Architecture**: Pre-trained transformer model with 12 layers, 768 hidden dimensions
- **Transfer Learning**: Leverages general language understanding and adapts it to domain-specific sentiment classification
- **Fine-tuning Strategy**: Uses a classification head on top of BERT's learned representations
- **Mean Pooling**: Aggregates token-level representations into sentence-level embeddings

## Dataset
- **Source**: Twitter Financial News Sentiment (Hugging Face)
- **Size**: ~4,000 training + 1,000 test + 1,200 validation samples
- **Classes**: 3 sentiment labels (Bearish, Bullish, Neutral)
- **Modality**: Short text tweets about financial markets

## Expected Outcomes
Higher accuracy than traditional ML methods due to BERT's contextual understanding and transformer architecture's ability to capture long-range dependencies.

## 1. Environment Setup and Dependencies

In [ ]:
# Deep Learning & PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Transformers & Pre-trained Models
from transformers import (
    AutoTokenizer,      # Tokenizer for input processing
    AutoModel,          # Pre-trained BERT model
    AutoConfig,         # Model configuration
    get_linear_schedule_with_warmup  # Learning rate scheduler
)

# Data Processing & Analysis
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

# Utilities
from tqdm import tqdm  # Progress bars
import json
import time
import os
from datetime import datetime

print("✓ PyTorch version:", torch.__version__)
print("✓ CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✓ GPU Device:", torch.cuda.get_device_name(0))

## 2. Configuration Settings

### Hyperparameter Definitions
- **model_name**: BERT variant (bert-base-uncased for English)
- **max_length**: Maximum token sequence length (64 is typical for short tweets)
- **batch_size**: Number of samples per gradient update
- **learning_rate**: Recommended 2e-5 for fine-tuning (lower than training from scratch)
- **num_epochs**: Typically 2-3 epochs prevent overfitting

In [ ]:
# Configuration Dictionary
CONFIG = {
    'model_name': 'bert-base-uncased',  # Base BERT model for English
    'max_length': 64,                    # Twitter/short text usually fit within 64 tokens
    'batch_size': 32,                    # Balance between memory and gradient stability
    'learning_rate': 2e-5,               # Standard for BERT fine-tuning (lower than pre-training)
    'num_epochs': 3,                     # Typically 2-3 epochs for good generalization
    'num_labels': 3,                     # Classes: Bearish (0), Bullish (1), Neutral (2)
    'seed': 42                           # Reproducibility
}

# Set random seeds for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# Enable CUDA reproducibility
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])
    torch.backends.cudnn.deterministic = True

print("✓ Configuration settings:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Data Loading and Preprocessing

In [ ]:
def load_twitter_data(data_dir='data'):
    """
    Load Twitter Financial News dataset from Hugging Face or cache
    
    Parameters:
    -----------
    data_dir : str
        Directory to store/load cached data
    
    Returns:
    --------
    tuple : (train_df, val_df, test_df, label_map)
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    
    label_map = {'Bearish': 0, 'Bullish': 1, 'Neutral': 2}
    
    # Check if data exists locally
    if not (data_dir / 'train.csv').exists():
        print("📥 Dataset not found locally, downloading from Hugging Face...")
        base_url = 'https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment/raw/main/'
        
        # Download full training data
        full_train_df = pd.read_csv(base_url + 'sent_train.csv')
        
        # Split into train and test
        train_df, test_df = train_test_split(
            full_train_df,
            test_size=0.2,
            random_state=CONFIG.get('seed', 42)
        )
        
        # Download validation set
        val_df = pd.read_csv(base_url + 'sent_valid.csv')
        
        # Save locally for faster future loading
        train_df.to_csv(data_dir / 'train.csv', index=False)
        val_df.to_csv(data_dir / 'val.csv', index=False)
        test_df.to_csv(data_dir / 'test.csv', index=False)
        
        print(f"✓ Downloaded and cached Twitter Financial News dataset")
    else:
        print(f"✓ Loading cached dataset from {data_dir}")
        train_df = pd.read_csv(data_dir / 'train.csv')
        val_df = pd.read_csv(data_dir / 'val.csv')
        test_df = pd.read_csv(data_dir / 'test.csv')
    
    print(f"\n  Train: {len(train_df):,} samples")
    print(f"  Val: {len(val_df):,} samples")
    print(f"  Test: {len(test_df):,} samples")
    print(f"  Classes: {list(label_map.keys())}")
    print(f"\n  Class Distribution (Train):")
    print(train_df['label'].value_counts())
    
    return train_df, val_df, test_df, label_map

print("✓ Data loading function defined")

In [ ]:
# Load the dataset
print("\n" + "="*70)
print("📊 LOADING TWITTER FINANCIAL NEWS DATASET")
print("="*70 + "\n")

train_df, val_df, test_df, label_map = load_twitter_data()

## 4. PyTorch Dataset Class

Custom Dataset class that handles:
- Tokenization using BERT's tokenizer
- Padding and truncation to fixed length
- Conversion to PyTorch tensors
- Attention masking to ignore padding tokens

In [ ]:
class TwitterSentimentDataset(Dataset):
    """
    PyTorch Dataset for Twitter sentiment classification
    
    Handles tokenization and tensor conversion for BERT input
    """
    
    def __init__(self, texts, labels, tokenizer, max_length):
        """
        Parameters:
        -----------
        texts : pd.Series or list
            Input text data
        labels : pd.Series or list
            Sentiment labels (0, 1, 2)
        tokenizer : transformers.AutoTokenizer
            BERT tokenizer for encoding text
        max_length : int
            Maximum sequence length
        """
        self.texts = texts.values if isinstance(texts, pd.Series) else texts
        self.labels = labels.values if isinstance(labels, pd.Series) else labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        """
        Get a single sample and encode it
        
        Returns:
        --------
        dict : Contains input_ids, attention_mask, and label tensors
        """
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        
        # Tokenize and encode
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',      # Pad to max_length
            truncation=True,           # Truncate if longer than max_length
            return_tensors='pt'        # Return PyTorch tensors
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

print("✓ TwitterSentimentDataset class defined")

## 5. BERT Model Architecture

### Architecture Details:
1. **BERT Encoder**: Pre-trained bidirectional transformer (frozen or fine-tuned)
2. **Mean Pooling**: Aggregates all token representations using attention mask
3. **Dropout Layer**: Regularization (p=0.1) to prevent overfitting
4. **Classification Head**: Linear layer mapping to 3 sentiment classes

### Mean Pooling Advantages:
- Captures all contextual information from all tokens
- More robust than [CLS] token alone
- Properly handles variable-length sequences

In [ ]:
class BERTMeanPoolingClassifier(nn.Module):
    """
    BERT with Mean Pooling for sentiment classification
    
    Architecture:
    - BERT encoder (pre-trained, fine-tunable)
    - Mean pooling over token representations
    - Dropout for regularization
    - Linear classification head
    """
    
    def __init__(self, model_name, num_labels):
        """
        Parameters:
        -----------
        model_name : str
            HuggingFace model identifier (e.g., 'bert-base-uncased')
        num_labels : int
            Number of classification labels
        """
        super().__init__()
        
        # Load pre-trained BERT configuration and model
        config = AutoConfig.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name, config=config)
        
        # Classification components
        self.dropout = nn.Dropout(0.1)  # Dropout regularization
        self.classifier = nn.Linear(config.hidden_size, num_labels)  # Output layer
        
    def forward(self, input_ids, attention_mask):
        """
        Forward pass through the model
        
        Parameters:
        -----------
        input_ids : torch.Tensor
            Tokenized input (batch_size, seq_length)
        attention_mask : torch.Tensor
            Attention mask for padding tokens (batch_size, seq_length)
        
        Returns:
        --------
        torch.Tensor : Logits (batch_size, num_labels)
        """
        # Get BERT outputs
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        # Extract last hidden state (all token representations)
        last_hidden = outputs.last_hidden_state  # (batch_size, seq_length, hidden_size)
        
        # MEAN POOLING: Average over all tokens (excluding padding)
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_hidden = torch.sum(last_hidden * attention_mask_expanded, dim=1)  # Sum valid tokens
        sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)  # Count valid tokens
        pooled_output = sum_hidden / sum_mask  # Average: (batch_size, hidden_size)
        
        # Apply dropout and classification
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)  # (batch_size, num_labels)
        
        return logits

print("✓ BERTMeanPoolingClassifier class defined")

## 6. Evaluation Function

In [ ]:
def evaluate(model, dataloader, device):
    """
    Evaluate model on validation or test set
    
    Parameters:
    -----------
    model : nn.Module
        Trained BERT classifier
    dataloader : DataLoader
        Validation/test dataloader
    device : torch.device
        Device to run evaluation on (cpu or cuda)
    
    Returns:
    --------
    tuple : (accuracy, precision, recall, f1, inference_time_ms)
    """
    model.eval()  # Set to evaluation mode
    all_preds = []
    all_labels = []
    
    start_time = time.time()
    
    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            # Forward pass
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    inference_time = time.time() - start_time
    num_samples = len(all_labels)
    inf_time_ms_per_sample = (inference_time / num_samples) * 1000 if num_samples > 0 else 0
    
    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds,
        average='weighted',
        zero_division=0
    )
    
    return acc, prec, rec, f1, inf_time_ms_per_sample

print("✓ Evaluation function defined")

## 7. Data Preparation and Dataloaders

In [ ]:
print("\n" + "="*70)
print("⚙️  PREPARING DATA AND MODEL")
print("="*70 + "\n")

# Initialize tokenizer
print("Loading BERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

# Create datasets
print("Creating datasets...")
train_dataset = TwitterSentimentDataset(
    train_df['text'], train_df['label'],
    tokenizer, CONFIG['max_length']
)

val_dataset = TwitterSentimentDataset(
    val_df['text'], val_df['label'],
    tokenizer, CONFIG['max_length']
)

test_dataset = TwitterSentimentDataset(
    test_df['text'], test_df['label'],
    tokenizer, CONFIG['max_length']
)

# Create dataloaders
print("Creating dataloaders...")
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'] * 2  # Larger batch for validation
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'] * 2
)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")
print(f"✓ Test batches: {len(test_loader)}")

## 8. Initialize Model, Optimizer, and Learning Rate Scheduler

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️  Using Device: {device}")

# Load model
print(f"\n📥 Loading pre-trained BERT Model ({CONFIG['model_name']})...")
model = BERTMeanPoolingClassifier(
    CONFIG['model_name'],
    CONFIG['num_labels']
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Model Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Optimizer with weight decay (L2 regularization)
optimizer = AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=0.01
)

# Learning rate scheduler with warmup
total_steps = len(train_loader) * CONFIG['num_epochs']
num_warmup_steps = int(total_steps * 0.1)  # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_steps
)

# Loss function
loss_fn = nn.CrossEntropyLoss()

print(f"\n⚙️  Training Configuration:")
print(f"  Learning Rate: {CONFIG['learning_rate']}")
print(f"  Total Steps: {total_steps:,}")
print(f"  Warmup Steps: {num_warmup_steps:,}")

## 9. Training Loop

### Training Strategy:
1. **Forward Pass**: Compute logits from BERT
2. **Loss Computation**: Calculate cross-entropy loss
3. **Backward Pass**: Compute gradients
4. **Gradient Clipping**: Prevent gradient explosion
5. **Optimization Step**: Update weights using AdamW
6. **Scheduler Step**: Update learning rate with warmup schedule
7. **Validation**: Evaluate on validation set after each epoch
8. **Model Checkpointing**: Save best model based on validation accuracy

In [ ]:
print("\n" + "="*70)
print("🚀 FINE-TUNING BERT FOR SENTIMENT CLASSIFICATION")
print("="*70 + "\n")

best_val_acc = 0
train_start_time = time.time()

# Training loop
for epoch in range(CONFIG['num_epochs']):
    model.train()  # Set to training mode
    total_train_loss = 0
    
    # Progress bar for training batches
    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']}"
    )
    
    for batch in progress_bar:
        # Move data to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping (prevent exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Optimizer step
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    # Calculate average training loss
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    val_acc, val_prec, val_rec, val_f1, _ = evaluate(model, val_loader, device)
    
    print(f"📊 Epoch {epoch+1} Complete:")
    print(f"   Train Loss: {avg_train_loss:.4f}")
    print(f"   Val Accuracy: {val_acc*100:.2f}%")
    print(f"   Val F1-Score: {val_f1*100:.2f}%\n")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_bert_mean.pt')
        print(f"   ✓ New best model saved (Acc: {val_acc*100:.2f}%)\n")

total_train_time = time.time() - train_start_time
print(f"✓ Training completed in {total_train_time:.1f} seconds\n")

## 10. Final Evaluation on Test Set

In [ ]:
print("\n" + "="*70)
print("🧪 FINAL EVALUATION ON TEST SET (UNSEEN DATA)")
print("="*70 + "\n")

# Load best model weights
print("Loading best model from checkpoint...")
model.load_state_dict(torch.load('best_bert_mean.pt'))

# Evaluate on test set
test_acc, test_prec, test_rec, test_f1, test_inf_ms = evaluate(model, test_loader, device)

# Get model size
model_size_mb = os.path.getsize('best_bert_mean.pt') / (1024 * 1024) if os.path.exists('best_bert_mean.pt') else 0

# Format parameters
def format_params(num):
    if num > 1e6:
        return f"{num/1e6:.1f}M"
    elif num > 1e3:
        return f"{num/1e3:.1f}K"
    return str(int(num))

# Display results
print("\n" + "="*70)
print("📈 FINAL RESULTS")
print("="*70)
print(f"\nModel Configuration:")
print(f"  Architecture: {CONFIG['model_name'].upper()} + Mean Pooling")
print(f"  Parameters: {format_params(total_params)}")
print(f"  Model Size: {model_size_mb:.2f} MB")
print(f"\nPerformance Metrics (Test Set):")
print(f"  Accuracy:  {test_acc*100:.2f}% ✓")
print(f"  Precision: {test_prec*100:.2f}% ✓")
print(f"  Recall:    {test_rec*100:.2f}% ✓")
print(f"  F1-Score:  {test_f1*100:.2f}% ✓")
print(f"\nEfficiency Metrics:")
print(f"  Training Time: {total_train_time:.1f}s")
print(f"  Inference: ~{test_inf_ms:.2f}ms per sample")
print(f"\n" + "="*70)

## 11. Visualization and Analysis

In [ ]:
# Get predictions for confusion matrix
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm)

# Classification report
class_names = ['Bearish', 'Bullish', 'Neutral']
print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(all_labels, all_preds, target_names=class_names))

## Summary and Conclusions

### Key Achievements:
1. **BERT Fine-tuning**: Successfully adapted pre-trained BERT for financial sentiment classification
2. **Mean Pooling Strategy**: Implemented attention-aware mean pooling for robust sentence representations
3. **Training Efficiency**: Used learning rate warmup and gradient clipping for stable optimization
4. **Model Checkpointing**: Saved best model based on validation accuracy to prevent overfitting

### Performance Advantages over Traditional ML:
- BERT captures bidirectional context through transformer attention mechanisms
- Pre-trained representations contain rich linguistic knowledge
- Better handling of complex semantic relationships in financial text

### Future Improvements:
- Experiment with other pooling strategies (CLS token, max pooling)
- Try larger BERT variants (bert-base-cased, RoBERTa, DistilBERT)
- Implement data augmentation techniques
- Fine-tune model with domain-specific financial BERT variants
- Ensemble multiple models for improved robustness